In [ ]:
import pandas as pd
datafile = './data/reviews_음식점.csv'
df = pd.read_csv(datafile)
df.head()

,category,place_name,reviewer,rating,date,text
0,맛집,삿포로 카스소바와 텐푸라 풍토. 하카타 나카스 점,gayoung Du,5,4개월 전,덴푸라바!!! 소수로 여행오셔서 다양한 튀김 맛보면서 반주 원하신다면 너무나 추천합...
1,맛집,삿포로 카스소바와 텐푸라 풍토. 하카타 나카스 점,히렁,5,수정일: 7개월 전,지인한테 추천받고간건데 저도 완전추천\n직원들부터 사장님 친절하고 재밌어요\n한국어...
2,맛집,삿포로 카스소바와 텐푸라 풍토. 하카타 나카스 점,워너블다니티,5,4개월 전,닭가슴살이 이렇게 육즙많고 부드러운지 몰랐어요\n비싸긴 했지만 정말정말정말 맛있는 ...
3,맛집,삿포로 카스소바와 텐푸라 풍토. 하카타 나카스 점,이이,5,6개월 전,전 예약을 해야한다해서 예약하고 갔는데 평일이라서 그런지 한가했어요~ 처음먹어본 따...
4,맛집,삿포로 카스소바와 텐푸라 풍토. 하카타 나카스 점,물고양이,5,6개월 전,예약없이 6:30에 들어갔습니다.\n곱창 매운 소바 꼭 드셔보시길\n오랫만에 매운거...


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175470 entries, 0 to 175469
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   category    175470 non-null  object
 1   place_name  175470 non-null  object
 2   reviewer    175455 non-null  object
 3   rating      175470 non-null  int64 
 4   date        175470 non-null  object
 5   text        147231 non-null  object
dtypes: int64(1), object(5)
memory usage: 8.0+ MB


In [ ]:
import datetime
import re
import pandas as pd

# 1. 파일 로드 및 기본 컬럼 정리
original_path = "data/reviews_음식점.csv"
df = pd.read_csv(original_path)

# 불필요 컬럼 제거 (가게명 필터링용 place_name과 날짜용 date는 절대 지우면 안 됩니다!)
df_cleaned = df.drop(columns=["category", "reviewer"]).dropna(subset=["text"])
df_cleaned = df_cleaned.drop_duplicates(subset=["place_name", "text"])


# 2. 🔥 [업그레이드] 구글 날짜 텍스트를 진짜 날짜(YYYY-MM-DD) 형식으로 역산하는 함수
def parse_google_date(date_text):
    now = datetime.datetime.now()
    date_text = str(date_text).strip()

    try:
        # 숫자 추출
        num_match = re.search(r"\d+", date_text)
        value = int(num_match.group()) if num_match else 0

        if "년 전" in date_text:
            calc_date = now - datetime.timedelta(days=value * 365)
        elif "개월 전" in date_text:
            calc_date = now - datetime.timedelta(days=value * 30)
        elif "주 전" in date_text:  # '주 전'도 간혹 나옵니다.
            calc_date = now - datetime.timedelta(weeks=value)
        elif "일 전" in date_text:
            calc_date = now - datetime.timedelta(days=value)
        elif "시간 전" in date_text:
            calc_date = now - datetime.timedelta(hours=value)
        elif "분 전" in date_text:
            calc_date = now - datetime.timedelta(minutes=value)
        else:
            # 그 외 (예: 방금 전 등)는 오늘 날짜
            calc_date = now

        # 나중에 연도별/월별 조회가 모두 가능하도록 표준 날짜 포맷(str)으로 리턴합니다.
        return calc_date.strftime("%Y-%m-%d")

    except Exception as e:
        # 에러 발생 시 안전하게 오늘 날짜로 방어
        return now.strftime("%Y-%m-%d")


# 진짜 날짜 형태의 임시 컬럼 생성
df_cleaned["calculated_date"] = df_cleaned["date"].apply(parse_google_date)

# 📊 대시보드 시각화와 필터링을 편하게 만들어 줄 핵심 컬럼 3개 생성!
df_cleaned["calculated_date"] = pd.to_datetime(df_cleaned["calculated_date"])
df_cleaned["year"] = df_cleaned["calculated_date"].dt.year  # 연도 (예: 2024)
df_cleaned["year_month"] = df_cleaned["calculated_date"].dt.to_period(
    "M"
)  # 년-월 (예: 2026-05)


# 3. 텍스트 노이즈 정제 및 길이 필터링
def clean_text(text):
    text = re.sub(r"\(구글 번역 제공\)|\(원문 보기\)", "", text)
    text = re.sub(
        r"[^\u2028\u2029ㄱ-ㅎㅏ-ㅣ가-힣0-9\s]", " ", text
    )  # 특수문자 제거
    text = re.sub(r"\s+", " ", text)
    return text.strip()


df_cleaned["text"] = df_cleaned["text"].apply(clean_text)
df_cleaned = df_cleaned[df_cleaned["text"].str.len() > 5]  # 5글자 이하 컷트

# 4. 리뷰가 20건 이상인 든든한 가게들만 필터링 (유령 가게 먼저 청소)
df_cleaned = df_cleaned.groupby("place_name").filter(lambda x: len(x) >= 20)

# 5. 3점 리뷰 분리 및 저장
df_final = df_cleaned[df_cleaned["rating"] != 3].copy()
df_only_3 = df_cleaned[df_cleaned["rating"] == 3].copy()

# 6. 최종 라벨링 (4, 5점은 1 / 1, 2점은 0)
df_final["label"] = (df_final["rating"] >= 4).astype(int)
df_final = df_final.drop(columns=["date"])

# 7. 최종 파일 저장
df_final.to_csv("output/reviews_음식점_학습용.csv", index=False, encoding="utf-8-sig")
df_only_3.to_csv(
    "output/reviews_음식점_3점_모음.csv", index=False, encoding="utf-8-sig"
)

print(f"🧹 전처리 및 필터링 완료!")
print(f"🏢 남은 맛집 수: {df_final['place_name'].nunique()}곳")
print(f"📊 최종 학습용 리뷰 개수: {len(df_final)}건")
print(
    f"📅 수집된 날짜 범위: {df_final['year_month'].min()} ~ {df_final['year_month'].max()}"
)

🧹 전처리 및 필터링 완료!
🏢 남은 맛집 수: 289곳
📊 최종 학습용 리뷰 개수: 130404건
📅 수집된 날짜 범위: 2011-06 ~ 2026-06


In [ ]:
df_jp = pd.read_csv("data/reviews_음식점_정제.csv")

# 2. 일본어 정규표현식 패턴 설정
# \u3040-\u309F: 히라가나 / \u30A0-\u30FF: 가타카나 / \u4E00-\u9FBF: 한자
ja_pattern = r"[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FBF]"

# 3. place_name에 일본어 패턴이 포함된 데이터만 필터링
# na=False를 넣어야 빈 값(NaN)이 있어도 에러가 나지 않습니다.
ja_shops_df = df_jp[df_jp["place_name"].str.contains(ja_pattern, regex=True, na=False)]

# 4. 중복을 제거한 순수한 일본어 가게 이름 목록 추출
ja_shop_list = ja_shops_df["place_name"].unique()

# 5. 결과 출력
print(f"📊 수집된 데이터 중 일본어가 포함된 가게는 총 {len(ja_shop_list)}군데입니다.")
print("-" * 50)
print("📋 일본어 가게 이름 목록:")
for idx, name in enumerate(ja_shop_list, 1):
    print(f"{idx}. {name}")

📊 수집된 데이터 중 일본어가 포함된 가게는 총 23군데입니다.
--------------------------------------------------
📋 일본어 가게 이름 목록:
1. 焼うお いし川 博多
2. 壺や
3. 居酒屋夢月希 할배집
4. 한국바 코리안바 Korean bar Style コリアンバー
5. 博多 晴家
6. 火星家
7. 喫茶 M & M
8. cafe’ cassette(カフェ カセット)
9. お酒とごはん おさななじみ
10. ネシガン
11. 宇奈とと×もんじゃココきよ大名店 우나토토 몬자 코코키요 다이묘점
12. 無限焼肉いいねいいね
13. 自社活魚車直接仕入れ 玄海亭本店
14. ホルモン鉄板鍋 幸家YUKIYA
15. もつ鍋専門店 博多もつ鍋専門店楽天地 離れ4階 ヨドバシ博多駅店
16. 四季膳あかり
17. 焼酎ダイニング 喜久屋
18. やひろ屋
19. 元祖トマトラーメンと辛麺と元祖トマトもつ鍋 三味(333) 十日えびす店
20. 博多バリ３酒場
21. 博多 灯（ともり）
22. 焼酎・旬彩料理CHIKO
23. 豊後炊き肉とお晩菜いっしょう


In [30]:
df_final["rating"].value_counts()

rating
5    87996
4    32385
1     5787
2     4317
Name: count, dtype: int64

In [ ]:
import pandas as pd

# 1. 현재 가지고 계신 '정제' 파일 불러오기
file_path = "data/reviews_음식점_정제.csv"
df = pd.read_csv(file_path)

print("=== 필터링 전 데이터 상태 ===")
print(f"총 리뷰 개수: {len(df)}건")
print(f"총 가게 개수: {df['place_name'].nunique()}곳")
print("-" * 40)

# 2. 가게별 리뷰 개수가 20개 이상인 데이터만 남기기
df_filtered = df.groupby("place_name").filter(lambda x: len(x) >= 20)

print("=== 필터링 후 최종 데이터 상태 ===")
print(f"최종 리뷰 개수: {len(df_filtered)}건")
print(f"최종 가게 개수: {df_filtered['place_name'].nunique()}곳")
print("-" * 40)

# 3. 20건 이상인 데이터만 같은 파일에 덮어쓰기
df_filtered.to_csv(file_path, index=False, encoding="utf-8-sig")


=== 🧹 필터링 전 데이터 상태 ===
총 리뷰 개수: 130485건
총 가게 개수: 297곳
----------------------------------------
=== ✨ 필터링 후 최종 데이터 상태 ===
최종 리뷰 개수: 130404건
최종 가게 개수: 289곳
----------------------------------------
✅ 리뷰 20건 미만인 가게들을 모두 제외하고 'output/reviews_음식점_정제.csv'에 저장했습니다!


In [ ]:
file_path = "data/reviews_음식점_정제.csv"
df = pd.read_csv(file_path)

df["label"] = (df["rating"] >= 4).astype(int)

print(f" 최종 긍정(1) 대 부정(0) 데이터 분포:\n{df['label'].value_counts()}")

df.to_csv("output/reviews_음식점_학습용.csv", index=False, encoding="utf-8-sig")

 최종 긍정(1) 대 부정(0) 데이터 분포:
label
1    120301
0     10103
Name: count, dtype: int64
